In [ ]:
import os 
os.getcwd()
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import os

In [ ]:

# Load the filez
data = loadmat('data.mat')
illumination = loadmat("illumination.mat")
pose=loadmat("pose.mat")
# See all the keys (variables stored in the .mat file)
print(f"data keys are {data.keys()}")
print(f"illumination keys are {illumination.keys()}")
print(f"pose keys are {pose.keys()}")

In [ ]:
data["face"].shape

In [ ]:
n=np.arange(1,201,1)
data_classified = {}
for i in n:
    imgs=[]
    for j in range(3):
        imgs.append(data["face"][:,:,3*i-3+j])
    data_classified[i]=imgs

In [ ]:
len(data_classified[1])

In [ ]:
n_2=np.arange(0,68,1)
n_2p=np.arange(0,13,1)
pose_classified={}
for i in n_2:
    imgs=[]
    for j in n_2p:
        imgs.append(pose["pose"][:,:,j,i])
    pose_classified[i]=imgs

In [ ]:
n_3=np.arange(0,68,1)
n_3i=np.arange(0,21,1)
illum_classified={}
for i in n_3:
    imgs=[]
    for j in n_3i:
        imgs.append(illumination["illum"][:,j,i])
    illum_classified[i]=imgs

### data visualization 

In [ ]:
fig, ax= plt.subplots(1,3)
ax[0].imshow(data_classified[90][0], cmap="gray") 
ax[0].set_title("neutral")
ax[0].axis("off")

ax[1].imshow(data_classified[90][1], cmap="gray")
ax[1].set_title("expression")
ax[1].axis("off")

ax[2].imshow(data_classified[90][2], cmap="gray")
ax[2].set_title("illumination")
ax[2].axis("off")

# data + labeling

In [ ]:
flattened_data = np.array([data["face"][:, :, i].flatten() for i in range(600)])
person_labels = np.repeat(np.arange(200), 3) #for task1

# for task2:
neutral_indices = list(range(0, 600, 3))
expression_indices = list(range(1, 600, 3))
binary_labels = np.zeros(600, dtype=int)
binary_labels[expression_indices] = 1


### MDA

In [ ]:
# Step 1: Prepare the data from the original face data
flattened_data = np.array([data["face"][:, :, i].flatten() for i in range(600)])

# Step 2: Calculate the mean and center the data
mean = np.mean(flattened_data, axis=0)
centered_data = flattened_data - mean

# Get the face dimensions for reshaping
face_height, face_width = 21, 24  # Based on description

# Step 3: Define classes by person identity (200 classes)
labels = np.zeros(flattened_data.shape[0], dtype=int)
for i in range(200):
    # Each person has 3 images (neutral, expression, illumination)
    # According to the description: face(:,:,3*n-2), face(:,:,3*n-1) and face(:,:,3*n)
    labels[3*i:3*(i+1)] = i

# Verify the label distribution
print(f"Number of classes: {len(np.unique(labels))}")
print(f"Images per class: {np.bincount(labels)[0]}")  # Should be 3

# Step 4: Calculate class means and scatter matrices
class_means = []
n_classes = 200  # 200 persons
total_samples = flattened_data.shape[0]

# Calculate mean for each class
for i in range(n_classes):
    class_data = flattened_data[labels == i]
    class_mean = np.mean(class_data, axis=0)
    class_means.append(class_mean)

# Overall mean (should be the same as 'mean' calculated earlier)
overall_mean = np.mean(flattened_data, axis=0)

# Between-class scatter matrix - using priors
S_B = np.zeros((flattened_data.shape[1], flattened_data.shape[1]))
for i in range(n_classes):
    n_samples = np.sum(labels == i)
    prior = n_samples / total_samples
    mean_diff = (class_means[i] - overall_mean).reshape(-1, 1)
    S_B += prior * np.dot(mean_diff, mean_diff.T)

# Within-class scatter matrix - using 1/n_i and priors
S_W = np.zeros((flattened_data.shape[1], flattened_data.shape[1]))
for i in range(n_classes):
    class_data = flattened_data[labels == i]
    n_samples = class_data.shape[0]
    prior = n_samples / total_samples
    class_centered = class_data - class_means[i]
    class_cov = np.dot(class_centered.T, class_centered) / n_samples
    S_W += prior * class_cov

# Step 5: Solve the generalized eigenvalue problem
# Using pseudo-inverse for stability
S_W_inv = np.linalg.pinv(S_W)
eigen_matrix = np.dot(S_W_inv, S_B)

# Calculate eigenvalues and eigenvectors
eigenvalues, eigenvectors_mda = np.linalg.eigh(eigen_matrix)

# Sort by eigenvalues in descending order
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors_mda = eigenvectors_mda[:, idx]

# Number of components to keep (at most n_classes-1)
# This could be very large (199), so let's limit it to a reasonable number
n_components_mda = min(n_classes-1, 200)  # Adjust this number as needed
eigenvectors_mda = eigenvectors_mda[:, :n_components_mda]

# Step 6: Project the data onto MDA space
mda_result = np.dot(centered_data, eigenvectors_mda)


In [ ]:
 #Step 7: Visualizations

# 7.1 Visualize original image vs MDA reconstruction
person_idx = 90  # Choose a person to visualize
image_type = 0   # 0=neutral, 1=expression, 2=illumination
sample_idx = person_idx * 3-3 + image_type

plt.figure(figsize=(12, 5))

# Original image
plt.subplot(1, 2, 1)
original_face = flattened_data[sample_idx].reshape(face_width, face_height)
plt.imshow(original_face, cmap='gray')
img_type_name = ["Neutral", "Expression", "Illumination"][image_type]
plt.title(f'Original Image - Person {person_idx} ({img_type_name})')
plt.axis('off')

# MDA reconstruction
projected_mda = np.dot(flattened_data[sample_idx] - mean, eigenvectors_mda)
reconstructed_mda = np.dot(projected_mda, eigenvectors_mda.T) + mean
reconstructed_face_mda = reconstructed_mda.reshape(face_width, face_height)

plt.subplot(1, 2, 2)
plt.imshow(reconstructed_face_mda, cmap='gray')
plt.title(f'MDA Reconstruction ({n_components_mda} components)')
plt.axis('off')

plt.suptitle('Original vs MDA-Reconstructed Image')
plt.tight_layout()
plt.show()



In [ ]:
# 7.3 Visualize the same person under different conditions
person_idx = 90  # Choose a person to visualize
plt.figure(figsize=(15, 8))
condition_names = ["Neutral", "Expression", "Illumination"]

# Show original and reconstructed for all three conditions
for i in range(3):
    # Original image
    plt.subplot(2, 3, i+1)
    img_idx = person_idx * 3 -3+ i
    orig_face = flattened_data[img_idx].reshape( face_width, face_height)
    plt.imshow(orig_face, cmap='gray')
    plt.title(f'Original ({condition_names[i]})')
    plt.axis('off')
    
    # MDA reconstruction
    plt.subplot(2, 3, i+4)
    projected = np.dot(flattened_data[img_idx] - mean, eigenvectors_mda)
    reconstructed = np.dot(projected, eigenvectors_mda.T) + mean
    recon_face = reconstructed.reshape( face_width, face_height)
    plt.imshow(recon_face, cmap='gray')
    plt.title(f'MDA Reconstruction')
    plt.axis('off')

plt.suptitle(f'Original vs MDA-Reconstructed Images for Person {person_idx}')
plt.tight_layout()
plt.show()


In [ ]:
# 7.4 Visualize a 2D embedding with first 2 MDA components
# Since we have 200 classes, plot just a subset for clarity
plt.figure(figsize=(12, 10))

# Choose a subset of people to visualize (e.g., first 20)
people_to_show = 20
colors = plt.cm.rainbow(np.linspace(0, 1, people_to_show))

for i in range(people_to_show):
    # Get data for this person
    indices = np.where(labels == i)[0]
    person_data = mda_result[indices]
    
    # Plot points for this person
    plt.scatter(
        person_data[:, 0],
        person_data[:, 1],
        color=colors[i],
        label=f"Person {i}",
        marker="o",
        s=80
    )
    
    # Optional: connect the points for this person to show the relationship
    # between neutral, expression, and illumination
    plt.plot(person_data[:, 0], person_data[:, 1], color=colors[i], alpha=0.5)

plt.title('MDA Projection - First 2 Components')
plt.xlabel('First Discriminant')
plt.ylabel('Second Discriminant')
# Use a smaller legend to fit all 20 people
plt.legend(fontsize='small', loc='upper right', bbox_to_anchor=(1.15, 1))
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # needed for 3D plotting
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

people_to_show = 20
colors = plt.cm.rainbow(np.linspace(0, 1, people_to_show))

for i in range(people_to_show):
    indices = np.where(labels == i)[0]
    person_data = mda_result[indices]

    ax.scatter(
        person_data[:, 0],  # First MDA component (X)
        person_data[:, 1],  # Second MDA component (Y)
        person_data[:, 2],  # Third MDA component (Z)
        color=colors[i],
        label=f"Person {i}",
        marker="o",
        s=80
    )

    # Optionally connect the points (line between 3 images)
    ax.plot(
        person_data[:, 0],
        person_data[:, 1],
        person_data[:, 2],
        color=colors[i],
        alpha=0.5
    )
ax.set_title("MDA Projection - First 3 Components")
ax.set_xlabel("1st Discriminant")
ax.set_ylabel("2nd Discriminant")
ax.set_zlabel("3rd Discriminant")
ax.legend(fontsize='small', loc='upper left', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()


# MDA Function

In [ ]:
import numpy as np

def compute_mda(data_matrix, labels, num_components=None):
    """
    Perform MDA (also known as LDA) on the given data.

    Parameters:
        data_matrix: np.ndarray of shape (n_samples, n_features)
        labels: array-like of shape (n_samples,)
        num_components: int or None — number of components to retain (must be ≤ n_classes - 1)

    Returns:
        mda_result: Projected data of shape (n_samples, num_components)
        components: Eigenvectors used for projection (n_features, num_components)
        eigenvalues: Corresponding eigenvalues
        overall_mean: Mean of the original data
    """
    n_samples, n_features = data_matrix.shape
    unique_classes = np.unique(labels)
    n_classes = len(unique_classes)

    # Step 1: Center the data
    overall_mean = np.mean(data_matrix, axis=0)
    centered_data = data_matrix - overall_mean

    # Step 2: Compute class means
    class_means = []
    for c in unique_classes:
        class_data = data_matrix[labels == c]
        class_mean = np.mean(class_data, axis=0)
        class_means.append(class_mean)

    # Step 3: Compute between-class scatter matrix (S_B)
    S_B = np.zeros((n_features, n_features))
    for i, c in enumerate(unique_classes):
        n_i = np.sum(labels == c)
        mean_diff = (class_means[i] - overall_mean).reshape(-1, 1)
        S_B += (n_i / n_samples) * (mean_diff @ mean_diff.T)

    # Step 4: Compute within-class scatter matrix (S_W)
    S_W = np.zeros((n_features, n_features))
    for i, c in enumerate(unique_classes):
        class_data = data_matrix[labels == c]
        n_i = class_data.shape[0]
        class_centered = class_data - class_means[i]
        S_W += (n_i / n_samples) * (class_centered.T @ class_centered) / n_i

    # Step 5: Solve generalized eigenvalue problem
    S_W_inv = np.linalg.pinv(S_W)
    eig_matrix = S_W_inv @ S_B
    eigenvalues, eigenvectors = np.linalg.eigh(eig_matrix)

    # Step 6: Sort eigenvectors by eigenvalue magnitude (descending)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Step 7: Limit number of components
    max_components = n_classes - 1
    if num_components is None or num_components > max_components:
        num_components = max_components
        print(f"Using max possible components for MDA: {num_components}")

    selected_components = eigenvectors[:, :num_components]
    mda_result = centered_data @ selected_components

    return mda_result, selected_components, eigenvalues[:num_components], overall_mean
